# フェーズ5：LLMによる解釈・施策提案

フェーズ2・3で作った来店・売上予測を、**店長が使える具体的な施策提案** に翻訳します。
これがMVP（分析パイプライン ＋ 解釈・提案プロンプト）の完成形です。

## 役割分担
- **数値予測（精度の中核）** … 機械学習モデル（`src/visit_forecast.py`）
- **解釈・施策提案** … LLMプロンプト（`src/llm_advisor.py`）

## 流れ
1. データ読み込み・モデル学習（フェーズ2のロジックを再利用）
2. 予測対象日の来店・売上を予測
3. 予測＋当日条件を構造化した「文脈」を作成
4. LLMプロンプトを組み立て
5. 施策提案を生成（APIキーがあればLLM、無ければルールベース下書き）

> **APIキーについて**：環境変数 `OPENAI_API_KEY` があれば実際のLLMが提案を生成します。
> 無い場合は、同じ文脈からルールベースの下書きを生成して動作を確認できます。
> いずれの場合も「数値予測はMLが担い、プロンプトは解釈・提案を担う」構成は同じです。

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd() / "src"))
from visit_forecast import train_models, predict_row, FEATURES
import llm_advisor
from llm_advisor import build_context, render_user_prompt, generate_advice, SYSTEM_PROMPT

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("セットアップ完了")

セットアップ完了


## 1. データ読み込み・モデル学習

フェーズ2のロジックを `visit_forecast` モジュールに切り出してあるので、数行で同じ予測パイプラインを再現できます。

In [2]:
csv_path = Path("data/retail_synthetic.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path, parse_dates=["date"])
else:
    from retail_synthetic_data import generate_retail_data
    df = generate_retail_data()

models = train_models(df, test_days=90)
df_feat = models.df_feat
print("学習完了。学習データ最終日:", models.train_end.date())
print("特徴量テーブル:", df_feat.shape)

学習完了。学習データ最終日: 2025-10-01
特徴量テーブル: (700, 37)


## 2. 予測対象日を選び、来店・売上を予測

検証期間から代表的な日（例：販促や悪天候など特徴のある日）を選んで予測します。
ここでは「特売 or ポイントデー」「雨の日」などを自動で拾います。

In [3]:
def make_advice_for(row: pd.Series, verbose: bool = True) -> dict:
    """1日分の特徴量行から、予測→文脈→施策提案までを行う."""
    pred = predict_row(models, row)

    # 比較基準：同曜日の平均、直近7日の平均（来店）
    dow_avg = df_feat[df_feat["dow"] == row["dow"]]["visitors"].mean()
    idx = df_feat.index[df_feat["date"] == row["date"]][0]
    recent = df_feat.iloc[max(0, idx - 7):idx]["visitors"].mean()

    context = build_context(row, pred, dow_avg_visitors=dow_avg, recent_avg_visitors=recent)
    advice = generate_advice(context)
    if verbose:
        print("=" * 70)
        print(f"対象日: {context['対象日']}（{context['曜日']}）")
        print(f"予測来店: {context['予測来店客数']:,}人 / 予測売上: {context['予測売上']:,}円")
        print("=" * 70)
    return {"context": context, "advice": advice, "pred": pred}


# 検証期間（学習データより後）の行を対象に、特徴のある日を選ぶ
test_rows = df_feat[df_feat["date"] > models.train_end].copy()

sample_sale = test_rows[test_rows["is_sale"] == 1].iloc[0]
sample_point = test_rows[test_rows["point_multiplier"] >= 5].iloc[0]
sample_rain = test_rows[test_rows["precipitation_mm"] >= 10].iloc[0]

print("選んだサンプル日:")
for label, r in [("特売日", sample_sale), ("ポイントデー", sample_point), ("雨の日", sample_rain)]:
    print(f"  {label}: {pd.Timestamp(r['date']).date()}")

選んだサンプル日:
  特売日: 2025-10-04
  ポイントデー: 2025-10-05
  雨の日: 2025-10-11


## 3. 文脈とプロンプトの確認

LLMに渡す「文脈（構造化データ）」と「プロンプト」を確認します。これが本MVPの主成果物です。

In [4]:
pred = predict_row(models, sample_sale)
dow_avg = df_feat[df_feat["dow"] == sample_sale["dow"]]["visitors"].mean()
idx = df_feat.index[df_feat["date"] == sample_sale["date"]][0]
recent = df_feat.iloc[max(0, idx - 7):idx]["visitors"].mean()
context = build_context(sample_sale, pred, dow_avg, recent)

print("【文脈（構造化データ）】")
for k, v in context.items():
    print(f"  {k}: {v}")

【文脈（構造化データ）】
  対象日: 2025-10-04
  曜日: 土
  予測来店客数: 1576
  予測売上: 4663586
  予測客単価: 2959
  曜日平均比: 1.22
  直近7日平均比: 1.8
  祝日: False
  週末: True
  給料日前後: False
  ポイント倍率: 1
  特売日: True
  クーポン配信: False
  天気: 晴れ
  最高気温: 27.0
  最低気温: 14.5
  降水量mm: 0.0
  SNS言及_前日: 675
  SNS言及_7日平均: 348
  SNS急増: True


In [5]:
print("===== システムプロンプト =====")
print(SYSTEM_PROMPT)
print("\n===== ユーザープロンプト =====")
print(render_user_prompt(context))

===== システムプロンプト =====
あなたは小売・サービス業の店舗運営を支援する熟練アドバイザーAIです。
ポイントシステムの購買データと、天候・人流・SNSをもとにした機械学習の来店・売上予測を受け取り、
店長が翌日の準備に使える、具体的で実行可能な助言を行います。

出力のルール:
- 数値予測は与えられた値を尊重し、勝手に作り変えない（解釈と行動提案に集中する）
- 次の見出しで簡潔にまとめる:
  【明日の見通し】予測と、その背景（なぜそうなるか）を2〜3文で
  【人員・シフト】来店予測に応じた人員配置の提案
  【発注・在庫】売上・天候・販促をふまえた商品準備の提案
  【販促・接客】ポイントデー/特売/クーポン/SNSをふまえた当日の打ち手
  【注意点】見落としやすいリスクや不確実性
- 専門用語は避け、店舗スタッフが理解できる平易な日本語で書く
- 根拠（曜日・天候・販促・SNSなど）を必ず添える

===== ユーザープロンプト =====
以下は明日の店舗の予測と条件です。これをもとに助言してください。

## 予測
- 対象日: 2025-10-04
- 曜日: 土
- 予測来店客数: 1576
- 予測売上: 4663586
- 予測客単価: 2959
- 曜日平均比: 1.22
- 直近7日平均比: 1.8

## 当日の条件
- 祝日: False
- 週末: True
- 給料日前後: False
- ポイント倍率: 1
- 特売日: True
- クーポン配信: False

## 天候（予報）
- 天気: 晴れ
- 最高気温: 27.0
- 最低気温: 14.5
- 降水量mm: 0.0

## SNS動向（前日まで）
- SNS言及_前日: 675
- SNS言及_7日平均: 348
- SNS急増: True


## 4. 施策提案の生成

`OPENAI_API_KEY` があればLLMが、無ければルールベースの下書きが提案を生成します。
3つの代表日（特売・ポイントデー・雨の日）について提案を出してみます。

In [6]:
mode = "LLM (OpenAI API)" if os.environ.get("OPENAI_API_KEY") else "ルールベース下書き（APIキー未設定）"
print(f"生成モード: {mode}\n")

for label, r in [("特売日", sample_sale), ("ポイントデー", sample_point), ("雨の日", sample_rain)]:
    res = make_advice_for(r)
    print(res["advice"])
    print()

生成モード: ルールベース下書き（APIキー未設定）

対象日: 2025-10-04（土）
予測来店: 1,576人 / 予測売上: 4,663,586円
【明日の見通し】
2025-10-04（土）の来店は約1,576人、売上は約4,663,586円の見込みです（曜日平均比 1.22）。
来店は増加傾向で、主な要因は週末・特売日・SNSで話題が急増です。

【人員・シフト】
来店増が見込まれるため、レジ・品出しを通常より増員し、ピーク時間帯のシフトを厚くしてください。

【発注・在庫】
来店・売上増に備え、主力商品と日配品を多めに発注。

【販促・接客】
特売目玉商品をエンド（売場端）に大きく展開し、関連商品を併せ買い提案／SNSで話題の商品・サービスを目立つ場所に配置し、当日もSNS発信。

【注意点】
SNS起因の来店は読みづらいので、品切れに即対応できる体制を。

※ これはルールベースの下書きです。OpenAI APIキーを設定すると、より自然で文脈に応じた提案をLLMが生成します。

対象日: 2025-10-05（日）
予測来店: 1,319人 / 予測売上: 3,960,361円
【明日の見通し】
2025-10-05（日）の来店は約1,319人、売上は約3,960,361円の見込みです（曜日平均比 1.08）。
来店は増加傾向で、主な要因は週末・特売日・ポイント5倍デーです。

【人員・シフト】
通常通りの人員配置で対応可能です。

【発注・在庫】
来店・売上増に備え、主力商品と日配品を多めに発注。

【販促・接客】
ポイント5倍を店頭・レジで告知し、会員カードの提示を促す／特売目玉商品をエンド（売場端）に大きく展開し、関連商品を併せ買い提案。

【注意点】
大きな不確実要因はなし。予測はあくまで目安として活用を。

※ これはルールベースの下書きです。OpenAI APIキーを設定すると、より自然で文脈に応じた提案をLLMが生成します。

対象日: 2025-10-11（土）
予測来店: 813人 / 予測売上: 2,156,096円
【明日の見通し】
2025-10-11（土）の来店は約813人、売上は約2,156,096円の見込みです（曜日平均比 0.63）。
来店は減少傾向で、主な要因は週末・雨予報です。

【人員・シフト】
来店は控えめ

## 5. 実運用での使い方（メモ）

実際のLLM提案を使うには、APIキーを設定してから本ノートブックを実行します。

```bash
export OPENAI_API_KEY="sk-..."   # 自分のキー
pip install openai               # 未インストールなら
```

設定後は `generate_advice(context)` が自動的にLLMを呼び出します（キーが無ければルールベースに自動フォールバック）。

## まとめ

- 機械学習の来店・売上予測を、店長向けの **具体的な施策提案** に翻訳するプロンプトを実装
- 予測（数値）はML、解釈・提案はLLMプロンプト、という役割分担を実現
- APIキーが無くてもルールベースで動作確認でき、キーを入れればそのままLLMに切替

### これでMVP（分析パイプライン ＋ 解釈・施策提案プロンプト）が一通り完成

### 今後の発展
- フェーズ4：会員個人単位の来店・離反予測（CRM・クーポン最適化）
- 実データ接続・複数店舗対応・DB連携・定期実行
- 施策提案の精緻化（在庫・人時の最適化、効果測定との連動）